# ModernBERT Multi-Task GECS Fine-Tune

This notebook is the locked Colab training path for the Week 7 push.

Backbone: `answerdotai/ModernBERT-base`

Training recipe:
- Segment-only input: `SegmentName + SegmentDescription`
- Multi-task heads: sector (11), group (55), industry (145)
- Mixed precision: bf16
- Optimizer: StableAdamW if available, else AdamW fallback
- Long-tail handling: distribution-balanced weighting on the industry head
- Checkpoints saved locally and downloaded at the end

This notebook is now set up for the real full training run by default. Model selection happens on a held-out dev split from the training data. The official test set is only evaluated once at the end of a full run.


In [ ]:
# 1. Environment setup
import os
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
REQUIRE_A100 = True
REQUIRE_BF16 = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA available:', torch.cuda.is_available())
gpu_name = None
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print('GPU:', gpu_name)
    props = torch.cuda.get_device_properties(0)
    print('VRAM GB:', round(props.total_memory / 1e9, 1))

!nvidia-smi
if REQUIRE_A100 and (gpu_name is None or 'A100' not in gpu_name):
    raise RuntimeError(f'Expected an A100 session for the locked recipe, got: {gpu_name}')
if REQUIRE_BF16 and not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
    raise RuntimeError('bf16 support is required. Reconnect until Colab gives you an A100-class session.')
!pip install -q -U transformers accelerate datasets scikit-learn evaluate safetensors sentencepiece pytorch-optimizer

In [ ]:
# 2. Local output directory inside the Colab session
RUN_NAME = 'modernbert_gecs_v2_full'
OUT_DIR = Path('/content') / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Saving artifacts to:', OUT_DIR)
print('This notebook now uses direct upload + direct download. No Drive mount required.')


In [ ]:
# 3. Load training data
# Preferred: upload task1_train.csv, task1_test.csv, and optionally gecs_taxonomy.json.
from google.colab import files

print('Upload task1_train.csv, task1_test.csv, and optionally gecs_taxonomy.json')
uploaded = files.upload()

TRAIN_PATH = 'task1_train.csv'
TEST_PATH = 'task1_test.csv'
TAXONOMY_PATH = 'gecs_taxonomy.json' if os.path.exists('gecs_taxonomy.json') else None

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print('train rows:', len(train_df))
print('test rows:', len(test_df))
print('train columns:', sorted(train_df.columns.tolist()))


In [ ]:
# 4. Build clean segment-only text and hierarchy labels
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def norm_code(value):
    return str(int(value)).zfill(8)

REQUIRED_SEGMENT_COLUMNS = ['SegmentName', 'SegmentDescription']
missing_train = [col for col in REQUIRED_SEGMENT_COLUMNS if col not in train_df.columns]
missing_test = [col for col in REQUIRED_SEGMENT_COLUMNS if col not in test_df.columns]
if missing_train or missing_test:
    raise RuntimeError(
        'This notebook is locked to segment-only training. Missing columns: '
        f'train={missing_train}, test={missing_test}. Re-export the cleaned segment-level CSVs before training.'
    )

def build_segment_text(df):
    seg = df['SegmentName'].fillna('').astype(str)
    desc = df['SegmentDescription'].fillna('').astype(str)
    text = (seg + ' ' + desc).str.replace(r'\s+', ' ', regex=True).str.strip()
    return text

for frame in (train_df, test_df):
    frame['industry_code'] = frame['mstar_code'].map(norm_code)
    frame['sector_code'] = frame['industry_code'].str[:3]
    frame['group_code'] = frame['industry_code'].str[:5]
    frame['segment_text'] = build_segment_text(frame)

train_df = train_df[train_df['segment_text'].str.len() > 0].copy()
test_df = test_df[test_df['segment_text'].str.len() > 0].copy()

le_sector = LabelEncoder().fit(pd.concat([train_df['sector_code'], test_df['sector_code']], axis=0))
le_group = LabelEncoder().fit(pd.concat([train_df['group_code'], test_df['group_code']], axis=0))
le_industry = LabelEncoder().fit(pd.concat([train_df['industry_code'], test_df['industry_code']], axis=0))

for frame in (train_df, test_df):
    frame['sector_idx'] = le_sector.transform(frame['sector_code'])
    frame['group_idx'] = le_group.transform(frame['group_code'])
    frame['industry_idx'] = le_industry.transform(frame['industry_code'])

N_SECTORS = len(le_sector.classes_)
N_GROUPS = len(le_group.classes_)
N_INDUSTRIES = len(le_industry.classes_)
print('Sector classes:', N_SECTORS)
print('Group classes:', N_GROUPS)
print('Industry classes:', N_INDUSTRIES)
print('Train rows after cleanup:', len(train_df))
print('Test rows after cleanup:', len(test_df))

try:
    train_fit_df, dev_df = train_test_split(
        train_df,
        test_size=0.10,
        random_state=SEED,
        stratify=train_df['industry_code'],
    )
except ValueError:
    train_fit_df, dev_df = train_test_split(
        train_df,
        test_size=0.10,
        random_state=SEED,
        shuffle=True,
    )

train_fit_df = train_fit_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print('Train-fit rows:', len(train_fit_df))
print('Dev rows:', len(dev_df))


In [ ]:
# 5. Run mode
# Keep this False for the real training run.
# Switch to True only when you explicitly want a short smoke test.
SMOKE_TEST = False
SMOKE_SAMPLES = 500
FULL_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 2
MIN_DEV_F1_DELTA = 0.001

if SMOKE_TEST:
    train_run_df = train_fit_df.sample(min(SMOKE_SAMPLES, len(train_fit_df)), random_state=SEED).copy()
    dev_run_df = dev_df.sample(min(SMOKE_SAMPLES, len(dev_df)), random_state=SEED).copy()
    test_run_df = test_df.copy()
    EPOCHS = 1
else:
    train_run_df = train_fit_df.copy()
    dev_run_df = dev_df.copy()
    test_run_df = test_df.copy()
    EPOCHS = FULL_EPOCHS

print('SMOKE_TEST:', SMOKE_TEST)
print('Run name:', RUN_NAME)
print('train_run rows:', len(train_run_df))
print('dev_run rows:', len(dev_run_df))
print('official test rows:', len(test_run_df))


In [ ]:
# 6. Tokenization
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'answerdotai/ModernBERT-base'
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

train_ds = Dataset.from_pandas(
    train_run_df[['segment_text', 'sector_idx', 'group_idx', 'industry_idx']].rename(columns={'segment_text': 'text'}),
    preserve_index=False,
)
dev_ds = Dataset.from_pandas(
    dev_run_df[['segment_text', 'sector_idx', 'group_idx', 'industry_idx']].rename(columns={'segment_text': 'text'}),
    preserve_index=False,
)
test_ds = Dataset.from_pandas(
    test_run_df[['segment_text', 'sector_idx', 'group_idx', 'industry_idx']].rename(columns={'segment_text': 'text'}),
    preserve_index=False,
)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=['text'])
dev_ds = dev_ds.map(tokenize_batch, batched=True, remove_columns=['text'])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=['text'])

columns = ['input_ids', 'attention_mask', 'sector_idx', 'group_idx', 'industry_idx']
train_ds.set_format(type='torch', columns=columns)
dev_ds.set_format(type='torch', columns=columns)
test_ds.set_format(type='torch', columns=columns)

print(train_ds)
print(dev_ds)
print(test_ds)


In [ ]:
# 7. Model, loss, and optimizer
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModel, get_cosine_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def effective_num_weights(labels, num_classes, beta=0.9999):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    eff_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.clip(eff_num, 1e-12, None)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

sector_weights = effective_num_weights(train_run_df['sector_idx'].to_numpy(), N_SECTORS)
group_weights = effective_num_weights(train_run_df['group_idx'].to_numpy(), N_GROUPS)
industry_weights = effective_num_weights(train_run_df['industry_idx'].to_numpy(), N_INDUSTRIES)

class DistributionBalancedCELoss(nn.Module):
    def __init__(self, class_weights, label_smoothing=0.02):
        super().__init__()
        self.register_buffer('class_weights', class_weights)
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        return F.cross_entropy(
            logits,
            targets,
            weight=self.class_weights,
            label_smoothing=self.label_smoothing,
        )

class MultiTaskModernBERT(nn.Module):
    def __init__(self, model_name, n_sectors, n_groups, n_industries, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.sector_head = nn.Linear(hidden, n_sectors)
        self.group_head = nn.Linear(hidden, n_groups)
        self.industry_head = nn.Linear(hidden, n_industries)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        return {
            'sector_logits': self.sector_head(pooled),
            'group_logits': self.group_head(pooled),
            'industry_logits': self.industry_head(pooled),
            'embedding': pooled,
        }

model = MultiTaskModernBERT(MODEL_NAME, N_SECTORS, N_GROUPS, N_INDUSTRIES).to(device)

alpha, beta_loss, gamma = 0.2, 0.3, 0.5
loss_sector = nn.CrossEntropyLoss(weight=sector_weights.to(device), label_smoothing=0.02)
loss_group = nn.CrossEntropyLoss(weight=group_weights.to(device), label_smoothing=0.02)
loss_industry = DistributionBalancedCELoss(industry_weights.to(device), label_smoothing=0.02)

def build_optimizer(model):
    encoder_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith('encoder.'):
            encoder_params.append(param)
        else:
            head_params.append(param)
    groups = [
        {'params': encoder_params, 'lr': 1e-5, 'weight_decay': 0.01},
        {'params': head_params, 'lr': 5e-4, 'weight_decay': 0.01},
    ]
    try:
        from pytorch_optimizer import StableAdamW
        optimizer = StableAdamW(groups)
        optimizer_name = 'StableAdamW'
    except Exception:
        optimizer = torch.optim.AdamW(groups)
        optimizer_name = 'AdamW fallback'
    return optimizer, optimizer_name

optimizer, optimizer_name = build_optimizer(model)
print('Optimizer:', optimizer_name)

BATCH_SIZE = 16
GRAD_ACCUM = 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = max(1, int(total_steps * 0.05))
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if REQUIRE_BF16 and not USE_BF16:
    raise RuntimeError('This recipe is locked to bf16. Reconnect to an A100 session before continuing.')
autocast_dtype = torch.bfloat16 if USE_BF16 else torch.float32
print('bf16 enabled:', USE_BF16)
print('total optimizer steps:', total_steps)


In [ ]:
# 8. Training loop with per-epoch checkpointing
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

best_macro_f1 = -1.0
history = []
epochs_without_improvement = 0

def run_eval(loader, eval_name='eval'):
    model.eval()
    all_sector_true, all_sector_pred = [], []
    all_group_true, all_group_pred = [], []
    all_industry_true, all_industry_pred = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=eval_name, leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sector_idx = batch['sector_idx'].to(device)
            group_idx = batch['group_idx'].to(device)
            industry_idx = batch['industry_idx'].to(device)
            with torch.autocast(device_type='cuda', dtype=autocast_dtype, enabled=USE_BF16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            sector_pred = outputs['sector_logits'].argmax(dim=-1)
            group_pred = outputs['group_logits'].argmax(dim=-1)
            industry_pred = outputs['industry_logits'].argmax(dim=-1)
            all_sector_true.extend(sector_idx.cpu().tolist())
            all_sector_pred.extend(sector_pred.cpu().tolist())
            all_group_true.extend(group_idx.cpu().tolist())
            all_group_pred.extend(group_pred.cpu().tolist())
            all_industry_true.extend(industry_idx.cpu().tolist())
            all_industry_pred.extend(industry_pred.cpu().tolist())
    true_codes = le_industry.inverse_transform(all_industry_true)
    pred_codes = le_industry.inverse_transform(all_industry_pred)
    return {
        'sector_acc': accuracy_score(all_sector_true, all_sector_pred),
        'group_acc': accuracy_score(all_group_true, all_group_pred),
        'industry_acc': accuracy_score(all_industry_true, all_industry_pred),
        'industry_macro_f1': f1_score(true_codes, pred_codes, average='macro', zero_division=0),
        'true_codes': true_codes.tolist(),
        'pred_codes': pred_codes.tolist(),
    }

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    started = time.time()
    progress = tqdm(train_loader, desc=f'train epoch {epoch + 1}/{EPOCHS}')
    for step, batch in enumerate(progress, start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sector_idx = batch['sector_idx'].to(device)
        group_idx = batch['group_idx'].to(device)
        industry_idx = batch['industry_idx'].to(device)
        with torch.autocast(device_type='cuda', dtype=autocast_dtype, enabled=USE_BF16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            sector_loss = loss_sector(outputs['sector_logits'], sector_idx)
            group_loss = loss_group(outputs['group_logits'], group_idx)
            industry_loss = loss_industry(outputs['industry_logits'], industry_idx)
            loss = alpha * sector_loss + beta_loss * group_loss + gamma * industry_loss
            loss = loss / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        progress.set_postfix({'loss': round(epoch_loss / step, 4)})

    metrics = run_eval(dev_loader, eval_name='dev')
    elapsed = time.time() - started
    epoch_record = {
        'epoch': epoch + 1,
        'train_loss': epoch_loss / max(1, len(train_loader)),
        'dev_sector_acc': metrics['sector_acc'],
        'dev_group_acc': metrics['group_acc'],
        'dev_industry_acc': metrics['industry_acc'],
        'dev_industry_macro_f1': metrics['industry_macro_f1'],
        'elapsed_s': elapsed,
    }
    history.append(epoch_record)
    print(json.dumps(epoch_record, indent=2))

    checkpoint_dir = OUT_DIR / f'epoch_{epoch + 1}'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), checkpoint_dir / 'model_state.pt')
    tokenizer.save_pretrained(checkpoint_dir)
    with open(checkpoint_dir / 'metrics.json', 'w', encoding='utf-8') as handle:
        json.dump(epoch_record, handle, indent=2)

    if metrics['industry_macro_f1'] > best_macro_f1 + MIN_DEV_F1_DELTA:
        best_macro_f1 = metrics['industry_macro_f1']
        epochs_without_improvement = 0
        torch.save(model.state_dict(), OUT_DIR / 'best_model_state.pt')
        with open(OUT_DIR / 'best_metrics.json', 'w', encoding='utf-8') as handle:
            json.dump(epoch_record, handle, indent=2)
        print('New best checkpoint saved.')
    else:
        epochs_without_improvement += 1
        print(f'No meaningful dev F1 improvement for {epochs_without_improvement} epoch(s).')
        if not SMOKE_TEST and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('Early stopping triggered.')
            break

with open(OUT_DIR / 'train_history.json', 'w', encoding='utf-8') as handle:
    json.dump(history, handle, indent=2)
print('Best macro F1:', round(best_macro_f1 * 100, 2))


In [ ]:
# 9. Final evaluation and artifact export
from collections import Counter
from google.colab import files

model.load_state_dict(torch.load(OUT_DIR / 'best_model_state.pt', map_location=device))

if SMOKE_TEST:
    smoke_summary = {
        'model_name': MODEL_NAME,
        'run_name': RUN_NAME,
        'smoke_test': True,
        'optimizer': optimizer_name,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation': GRAD_ACCUM,
        'max_len': MAX_LEN,
        'best_dev_macro_f1': best_macro_f1,
        'notes': 'Smoke test only. Official test set intentionally not evaluated.',
    }
    with open(OUT_DIR / 'smoke_summary.json', 'w', encoding='utf-8') as handle:
        json.dump(smoke_summary, handle, indent=2)
    print(json.dumps(smoke_summary, indent=2))
    smoke_zip = f'/content/{RUN_NAME}_smoke_outputs.zip'
    !zip -qr "$smoke_zip" "$OUT_DIR"
    files.download(smoke_zip)
else:
    final_metrics = run_eval(test_loader, eval_name='official_test')
    true_codes = final_metrics['true_codes']
    pred_codes = final_metrics['pred_codes']
    counts = Counter(true_codes)
    top10 = [code for code, _ in counts.most_common(10)]
    top10_f1 = f1_score(true_codes, pred_codes, average=None, labels=top10, zero_division=0)
    top10_pass = int(sum(score > 0.85 for score in top10_f1))

    summary = {
        'model_name': MODEL_NAME,
        'run_name': RUN_NAME,
        'smoke_test': False,
        'optimizer': optimizer_name,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation': GRAD_ACCUM,
        'max_len': MAX_LEN,
        'sector_acc': final_metrics['sector_acc'],
        'group_acc': final_metrics['group_acc'],
        'industry_acc': final_metrics['industry_acc'],
        'industry_macro_f1': final_metrics['industry_macro_f1'],
        'best_dev_macro_f1': best_macro_f1,
        'top10_pass': top10_pass,
        'top10_breakdown': [
            {'code': code, 'f1': float(score), 'support': counts[code]}
            for code, score in zip(top10, top10_f1)
        ],
        'loss_weights': {'sector': alpha, 'group': beta_loss, 'industry': gamma},
        'notes': 'Segment-only input, bf16, multi-task ModernBERT, DB-style industry weighting.',
    }

    with open(OUT_DIR / 'final_summary.json', 'w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2)

    np.save(OUT_DIR / 'industry_classes.npy', le_industry.classes_)
    np.save(OUT_DIR / 'group_classes.npy', le_group.classes_)
    np.save(OUT_DIR / 'sector_classes.npy', le_sector.classes_)
    pd.DataFrame({'true_code': true_codes, 'pred_code': pred_codes}).to_csv(OUT_DIR / 'test_predictions.csv', index=False)

    print(json.dumps(summary, indent=2))
    final_zip = f'/content/{RUN_NAME}_full_outputs.zip'
    !zip -qr "$final_zip" "$OUT_DIR"
    files.download(final_zip)


In [ ]:
# 10. ModernBERT v2 run notes
RUN_NOTES = {
    'model_name': MODEL_NAME,
    'run_name': RUN_NAME,
    'full_epochs': FULL_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_dev_f1_delta': MIN_DEV_F1_DELTA,
    'input_policy': 'SegmentName + SegmentDescription only',
    'precision': 'bf16 on A100',
    'next_step_if_v2_underperforms': 'Use logits as ensemble features with V13/RAC/GECS-anchor signals.',
}
print(json.dumps(RUN_NOTES, indent=2))
